<a href="https://colab.research.google.com/github/Emilycchi/googlecolabproject/blob/main/getscrimdata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import datetime
import json
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

# All relevant Game-Id's
SERIES_IDS_TO_PULL = ["2719263", "2719283", "2719315", "2719327"]
GameNumber = len(SERIES_IDS_TO_PULL)

# GRID-API key
headers = {
    "x-api-key": ""
}

# Team ID
TeamID = "mCon esports"

# Uses the users google account to open and write sheets
gc = gspread.authorize(creds)

# Opens the relevant google sheets document(use key to define which sheet)
sh = gc.open_by_key('')
wks = sh.worksheet('RawData')

# Goes through this big for loop once for every game
# Downloads and saves the relevant game files temporarily
for x in range(GameNumber):
    GameDownload = requests.get("https://api.grid.gg/file-download/end-state/grid/series/"+ SERIES_IDS_TO_PULL[x],
                                headers = headers)
    GamesumDownload = requests.get("https://api.grid.gg/file-download/end-state/riot/series/"
                                   + SERIES_IDS_TO_PULL[x]+ "/games/1/summary",
                                headers = headers)
    GamedetDownload = requests.get("https://api.grid.gg/file-download/end-state/riot/series/"
                                   + SERIES_IDS_TO_PULL[x]+"/games/{game number}/details",
                                headers = headers)

    # Create relevant variables to store relevant data temporarily
    game1 = GameDownload.json()
    game1sum = GamesumDownload.json()
    game1det = GamedetDownload.json()
    Top = ['Top']
    Jungle = ['Jgl']
    Mid = ['Mid']
    Bot = ['Bot']
    Supp = ['Supp']
    Draft = []
    Side = 0
    Datetime = game1["updatedAt"].split('T')
    Date = Datetime[0]

    # Saves the drafted champs, creates a blank draft if the draftactions don't add up to 20
    if len(game1["games"][0]["draftActions"]) == 20:
      for y in range(len(game1["games"][0]["draftActions"])):
        Draft.append(game1["games"][0]["draftActions"][y]["draftable"]["name"])
    while len(Draft)<20:
      Draft.append("-")

    # If else is used to identify which side the relevant team played(which data to pull)
    # Checks the time of the scrim and appends to variables
    if game1["teams"][0]["name"] == TeamID:
        Top.extend(("Blue", Date))
        Jungle.extend(("Blue", Date))
        Mid.extend(("Blue", Date))
        Bot.extend(("Blue", Date))
        Supp.extend(("Blue", Date))
        Side = "Blue"
    else:
        Top.extend(("Red", Date))
        Jungle.extend(("Red", Date))
        Mid.extend(("Red", Date))
        Bot.extend(("Red", Date))
        Supp.extend(("Red", Date))
        Side = "Red"

    # Checks whether the scrim was won or lost and appends to variables
    for z in range(2):
        if game1["teams"][z]["name"] == TeamID:
            if game1["teams"][z]["score"] == 1:   # needs work, output is boolean, needed to pivot
                Top.append("1")
                Jungle.append("1")
                Mid.append("1")
                Bot.append("1")
                Supp.append("1")
            else:
                Top.append("0")
                Jungle.append("0")
                Mid.append("0")
                Bot.append("0")
                Supp.append("0")

    # Appends the relevant champs to the role(both teams). Only works if players are ordered
    Top.extend((Draft[0], Draft[1], Draft[6], Draft[7]))
    Jungle.extend((Draft[2], Draft[3], Draft[9], Draft[8]))
    Mid.extend((Draft[4], Draft[5], Draft[10], Draft[11]))
    Bot.extend((Draft[13], Draft[12], Draft[17], Draft[16]))
    Supp.extend((Draft[15], Draft[14], Draft[18], Draft[19]))

    # Appends all the desired stats
    if Side == "Blue":
        Top.extend((game1sum["participants"][0]["championName"], game1sum["teams"][0]["objectives"]["baron"]["kills"],
                    game1sum["teams"][0]["objectives"]["dragon"]["kills"],
                    game1sum["teams"][0]["objectives"]["horde"]["kills"],
                    game1sum["teams"][0]["objectives"]["inhibitor"]["kills"],
                    game1sum["teams"][0]["objectives"]["riftHerald"]["kills"],
                    game1sum["teams"][0]["objectives"]["tower"]["kills"],
                    str(datetime.timedelta(seconds=game1sum["participants"][0]["challenges"]["gameLength"])),
                    game1sum["participants"][0]["challenges"]["turretPlatesTaken"],
                    game1sum["participants"][0]["challenges"]["teamDamagePercentage"],
                    game1sum["participants"][0]["magicDamageDealtToChampions"],
                    game1sum["participants"][0]["physicalDamageDealtToChampions"],
                    game1sum["participants"][0]["totalDamageDealtToChampions"],
                    game1sum["participants"][0]["goldEarned"], game1sum["participants"][0]["kills"],
                    game1sum["participants"][0]["deaths"],
                    game1sum["participants"][0]["assists"], game1sum["participants"][0]["visionScore"],
                    game1sum["participants"][0]["challenges"]["controlWardsPlaced"],
                    game1sum["participants"][0]["missions"]["Missions_CreepScore"],
                    game1sum["participants"][5]["championName"],
                    game1sum["participants"][5]["challenges"]["turretPlatesTaken"],
                    game1sum["teams"][1]["objectives"]["baron"]["kills"],
                    game1sum["teams"][1]["objectives"]["dragon"]["kills"],
                    game1sum["teams"][1]["objectives"]["horde"]["kills"],
                    game1sum["teams"][1]["objectives"]["inhibitor"]["kills"],
                    game1sum["teams"][1]["objectives"]["riftHerald"]["kills"],
                    game1sum["teams"][1]["objectives"]["tower"]["kills"], game1sum["gameVersion"]))
        Jungle.extend((
                      game1sum["participants"][1]["championName"], game1sum["teams"][0]["objectives"]["baron"]["kills"],
                      game1sum["teams"][0]["objectives"]["dragon"]["kills"],
                      game1sum["teams"][0]["objectives"]["horde"]["kills"],
                      game1sum["teams"][0]["objectives"]["inhibitor"]["kills"],
                      game1sum["teams"][0]["objectives"]["riftHerald"]["kills"],
                      game1sum["teams"][0]["objectives"]["tower"]["kills"],
                      str(datetime.timedelta(seconds=game1sum["participants"][0]["challenges"]["gameLength"])),
                      game1sum["participants"][1]["challenges"]["turretPlatesTaken"],
                      game1sum["participants"][1]["challenges"]["teamDamagePercentage"],
                      game1sum["participants"][1]["magicDamageDealtToChampions"],
                      game1sum["participants"][1]["physicalDamageDealtToChampions"],
                      game1sum["participants"][1]["totalDamageDealtToChampions"],
                      game1sum["participants"][1]["goldEarned"], game1sum["participants"][1]["kills"],
                      game1sum["participants"][1]["deaths"],
                      game1sum["participants"][1]["assists"], game1sum["participants"][1]["visionScore"],
                      game1sum["participants"][1]["challenges"]["controlWardsPlaced"],
                      game1sum["participants"][1]["missions"]["Missions_CreepScore"],
                      game1sum["participants"][6]["championName"],
                      game1sum["participants"][6]["challenges"]["turretPlatesTaken"],
                      game1sum["teams"][1]["objectives"]["baron"]["kills"],
                      game1sum["teams"][1]["objectives"]["dragon"]["kills"],
                      game1sum["teams"][1]["objectives"]["horde"]["kills"],
                      game1sum["teams"][1]["objectives"]["inhibitor"]["kills"],
                      game1sum["teams"][1]["objectives"]["riftHerald"]["kills"],
                      game1sum["teams"][1]["objectives"]["tower"]["kills"], game1sum["gameVersion"]))
        Mid.extend((game1sum["participants"][2]["championName"], game1sum["teams"][0]["objectives"]["baron"]["kills"],
                    game1sum["teams"][0]["objectives"]["dragon"]["kills"],
                    game1sum["teams"][0]["objectives"]["horde"]["kills"],
                    game1sum["teams"][0]["objectives"]["inhibitor"]["kills"],
                    game1sum["teams"][0]["objectives"]["riftHerald"]["kills"],
                    game1sum["teams"][0]["objectives"]["tower"]["kills"],
                    str(datetime.timedelta(seconds=game1sum["participants"][0]["challenges"]["gameLength"])),
                    game1sum["participants"][2]["challenges"]["turretPlatesTaken"],
                    game1sum["participants"][2]["challenges"]["teamDamagePercentage"],
                    game1sum["participants"][2]["magicDamageDealtToChampions"],
                    game1sum["participants"][2]["physicalDamageDealtToChampions"],
                    game1sum["participants"][2]["totalDamageDealtToChampions"],
                    game1sum["participants"][2]["goldEarned"], game1sum["participants"][2]["kills"],
                    game1sum["participants"][2]["deaths"],
                    game1sum["participants"][2]["assists"], game1sum["participants"][2]["visionScore"],
                    game1sum["participants"][2]["challenges"]["controlWardsPlaced"],
                    game1sum["participants"][2]["missions"]["Missions_CreepScore"],
                    game1sum["participants"][7]["championName"],
                    game1sum["participants"][7]["challenges"]["turretPlatesTaken"],
                    game1sum["teams"][1]["objectives"]["baron"]["kills"],
                    game1sum["teams"][1]["objectives"]["dragon"]["kills"],
                    game1sum["teams"][1]["objectives"]["horde"]["kills"],
                    game1sum["teams"][1]["objectives"]["inhibitor"]["kills"],
                    game1sum["teams"][1]["objectives"]["riftHerald"]["kills"],
                    game1sum["teams"][1]["objectives"]["tower"]["kills"], game1sum["gameVersion"]))
        Bot.extend((game1sum["participants"][3]["championName"], game1sum["teams"][0]["objectives"]["baron"]["kills"],
                    game1sum["teams"][0]["objectives"]["dragon"]["kills"],
                    game1sum["teams"][0]["objectives"]["horde"]["kills"],
                    game1sum["teams"][0]["objectives"]["inhibitor"]["kills"],
                    game1sum["teams"][0]["objectives"]["riftHerald"]["kills"],
                    game1sum["teams"][0]["objectives"]["tower"]["kills"],
                    str(datetime.timedelta(seconds=game1sum["participants"][0]["challenges"]["gameLength"])),
                    game1sum["participants"][3]["challenges"]["turretPlatesTaken"],
                    game1sum["participants"][3]["challenges"]["teamDamagePercentage"],
                    game1sum["participants"][3]["magicDamageDealtToChampions"],
                    game1sum["participants"][3]["physicalDamageDealtToChampions"],
                    game1sum["participants"][3]["totalDamageDealtToChampions"],
                    game1sum["participants"][3]["goldEarned"], game1sum["participants"][3]["kills"],
                    game1sum["participants"][3]["deaths"],
                    game1sum["participants"][3]["assists"], game1sum["participants"][3]["visionScore"],
                    game1sum["participants"][3]["challenges"]["controlWardsPlaced"],
                    game1sum["participants"][3]["missions"]["Missions_CreepScore"],
                    game1sum["participants"][8]["championName"],
                    game1sum["participants"][8]["challenges"]["turretPlatesTaken"],
                    game1sum["teams"][1]["objectives"]["baron"]["kills"],
                    game1sum["teams"][1]["objectives"]["dragon"]["kills"],
                    game1sum["teams"][1]["objectives"]["horde"]["kills"],
                    game1sum["teams"][1]["objectives"]["inhibitor"]["kills"],
                    game1sum["teams"][1]["objectives"]["riftHerald"]["kills"],
                    game1sum["teams"][1]["objectives"]["tower"]["kills"], game1sum["gameVersion"]))
        Supp.extend((game1sum["participants"][4]["championName"], game1sum["teams"][0]["objectives"]["baron"]["kills"],
                     game1sum["teams"][0]["objectives"]["dragon"]["kills"],
                     game1sum["teams"][0]["objectives"]["horde"]["kills"],
                     game1sum["teams"][0]["objectives"]["inhibitor"]["kills"],
                     game1sum["teams"][0]["objectives"]["riftHerald"]["kills"],
                     game1sum["teams"][0]["objectives"]["tower"]["kills"],
                     str(datetime.timedelta(seconds=game1sum["participants"][0]["challenges"]["gameLength"])),
                     game1sum["participants"][4]["challenges"]["turretPlatesTaken"],
                     game1sum["participants"][4]["challenges"]["teamDamagePercentage"],
                     game1sum["participants"][4]["magicDamageDealtToChampions"],
                     game1sum["participants"][4]["physicalDamageDealtToChampions"],
                     game1sum["participants"][4]["totalDamageDealtToChampions"],
                     game1sum["participants"][4]["goldEarned"], game1sum["participants"][4]["kills"],
                     game1sum["participants"][4]["deaths"],
                     game1sum["participants"][4]["assists"], game1sum["participants"][4]["visionScore"],
                     game1sum["participants"][4]["challenges"]["controlWardsPlaced"],
                     game1sum["participants"][4]["missions"]["Missions_CreepScore"],
                     game1sum["participants"][9]["championName"],
                     game1sum["participants"][9]["challenges"]["turretPlatesTaken"],
                     game1sum["teams"][1]["objectives"]["baron"]["kills"],
                     game1sum["teams"][1]["objectives"]["dragon"]["kills"],
                     game1sum["teams"][1]["objectives"]["horde"]["kills"],
                     game1sum["teams"][1]["objectives"]["inhibitor"]["kills"],
                     game1sum["teams"][1]["objectives"]["riftHerald"]["kills"],
                     game1sum["teams"][1]["objectives"]["tower"]["kills"], game1sum["gameVersion"]))
    else:
        Top.extend((game1sum["participants"][5]["championName"], game1sum["teams"][1]["objectives"]["baron"]["kills"],
                    game1sum["teams"][1]["objectives"]["dragon"]["kills"],
                    game1sum["teams"][1]["objectives"]["horde"]["kills"],
                    game1sum["teams"][1]["objectives"]["inhibitor"]["kills"],
                    game1sum["teams"][1]["objectives"]["riftHerald"]["kills"],
                    game1sum["teams"][1]["objectives"]["tower"]["kills"],
                    str(datetime.timedelta(seconds=game1sum["participants"][0]["challenges"]["gameLength"])),
                    game1sum["participants"][5]["challenges"]["turretPlatesTaken"],
                    game1sum["participants"][5]["challenges"]["teamDamagePercentage"],
                    game1sum["participants"][5]["magicDamageDealtToChampions"],
                    game1sum["participants"][5]["physicalDamageDealtToChampions"],
                    game1sum["participants"][5]["totalDamageDealtToChampions"],
                    game1sum["participants"][5]["goldEarned"], game1sum["participants"][5]["kills"],
                    game1sum["participants"][5]["deaths"],
                    game1sum["participants"][5]["assists"], game1sum["participants"][5]["visionScore"],
                    game1sum["participants"][5]["challenges"]["controlWardsPlaced"],
                    game1sum["participants"][5]["missions"]["Missions_CreepScore"],
                    game1sum["participants"][0]["championName"],
                    game1sum["participants"][0]["challenges"]["turretPlatesTaken"],
                    game1sum["teams"][0]["objectives"]["baron"]["kills"],
                    game1sum["teams"][0]["objectives"]["dragon"]["kills"],
                    game1sum["teams"][0]["objectives"]["horde"]["kills"],
                    game1sum["teams"][0]["objectives"]["inhibitor"]["kills"],
                    game1sum["teams"][0]["objectives"]["riftHerald"]["kills"],
                    game1sum["teams"][0]["objectives"]["tower"]["kills"], game1sum["gameVersion"]))
        Jungle.extend((
                      game1sum["participants"][6]["championName"], game1sum["teams"][1]["objectives"]["baron"]["kills"],
                      game1sum["teams"][1]["objectives"]["dragon"]["kills"],
                      game1sum["teams"][1]["objectives"]["horde"]["kills"],
                      game1sum["teams"][1]["objectives"]["inhibitor"]["kills"],
                      game1sum["teams"][1]["objectives"]["riftHerald"]["kills"],
                      game1sum["teams"][1]["objectives"]["tower"]["kills"],
                      str(datetime.timedelta(seconds=game1sum["participants"][0]["challenges"]["gameLength"])),
                      game1sum["participants"][6]["challenges"]["turretPlatesTaken"],
                      game1sum["participants"][6]["challenges"]["teamDamagePercentage"],
                      game1sum["participants"][6]["magicDamageDealtToChampions"],
                      game1sum["participants"][6]["physicalDamageDealtToChampions"],
                      game1sum["participants"][6]["totalDamageDealtToChampions"],
                      game1sum["participants"][6]["goldEarned"], game1sum["participants"][6]["kills"],
                      game1sum["participants"][6]["deaths"],
                      game1sum["participants"][6]["assists"], game1sum["participants"][6]["visionScore"],
                      game1sum["participants"][6]["challenges"]["controlWardsPlaced"],
                      game1sum["participants"][6]["missions"]["Missions_CreepScore"],
                      game1sum["participants"][1]["championName"],
                      game1sum["participants"][1]["challenges"]["turretPlatesTaken"],
                      game1sum["teams"][0]["objectives"]["baron"]["kills"],
                      game1sum["teams"][0]["objectives"]["dragon"]["kills"],
                      game1sum["teams"][0]["objectives"]["horde"]["kills"],
                      game1sum["teams"][0]["objectives"]["inhibitor"]["kills"],
                      game1sum["teams"][0]["objectives"]["riftHerald"]["kills"],
                      game1sum["teams"][0]["objectives"]["tower"]["kills"], game1sum["gameVersion"]))
        Mid.extend((game1sum["participants"][7]["championName"], game1sum["teams"][1]["objectives"]["baron"]["kills"],
                    game1sum["teams"][1]["objectives"]["dragon"]["kills"],
                    game1sum["teams"][1]["objectives"]["horde"]["kills"],
                    game1sum["teams"][1]["objectives"]["inhibitor"]["kills"],
                    game1sum["teams"][1]["objectives"]["riftHerald"]["kills"],
                    game1sum["teams"][1]["objectives"]["tower"]["kills"],
                    str(datetime.timedelta(seconds=game1sum["participants"][0]["challenges"]["gameLength"])),
                    game1sum["participants"][7]["challenges"]["turretPlatesTaken"],
                    game1sum["participants"][7]["challenges"]["teamDamagePercentage"],
                    game1sum["participants"][7]["magicDamageDealtToChampions"],
                    game1sum["participants"][7]["physicalDamageDealtToChampions"],
                    game1sum["participants"][7]["totalDamageDealtToChampions"],
                    game1sum["participants"][7]["goldEarned"], game1sum["participants"][7]["kills"],
                    game1sum["participants"][7]["deaths"],
                    game1sum["participants"][7]["assists"], game1sum["participants"][7]["visionScore"],
                    game1sum["participants"][7]["challenges"]["controlWardsPlaced"],
                    game1sum["participants"][7]["missions"]["Missions_CreepScore"],
                    game1sum["participants"][2]["championName"],
                    game1sum["participants"][2]["challenges"]["turretPlatesTaken"],
                    game1sum["teams"][0]["objectives"]["baron"]["kills"],
                    game1sum["teams"][0]["objectives"]["dragon"]["kills"],
                    game1sum["teams"][0]["objectives"]["horde"]["kills"],
                    game1sum["teams"][0]["objectives"]["inhibitor"]["kills"],
                    game1sum["teams"][0]["objectives"]["riftHerald"]["kills"],
                    game1sum["teams"][0]["objectives"]["tower"]["kills"], game1sum["gameVersion"]))
        Bot.extend((game1sum["participants"][8]["championName"], game1sum["teams"][1]["objectives"]["baron"]["kills"],
                    game1sum["teams"][1]["objectives"]["dragon"]["kills"],
                    game1sum["teams"][1]["objectives"]["horde"]["kills"],
                    game1sum["teams"][1]["objectives"]["inhibitor"]["kills"],
                    game1sum["teams"][1]["objectives"]["riftHerald"]["kills"],
                    game1sum["teams"][1]["objectives"]["tower"]["kills"],
                    str(datetime.timedelta(seconds=game1sum["participants"][0]["challenges"]["gameLength"])),
                    game1sum["participants"][8]["challenges"]["turretPlatesTaken"],
                    game1sum["participants"][8]["challenges"]["teamDamagePercentage"],
                    game1sum["participants"][8]["magicDamageDealtToChampions"],
                    game1sum["participants"][8]["physicalDamageDealtToChampions"],
                    game1sum["participants"][8]["totalDamageDealtToChampions"],
                    game1sum["participants"][8]["goldEarned"], game1sum["participants"][8]["kills"],
                    game1sum["participants"][8]["deaths"],
                    game1sum["participants"][8]["assists"], game1sum["participants"][8]["visionScore"],
                    game1sum["participants"][8]["challenges"]["controlWardsPlaced"],
                    game1sum["participants"][8]["missions"]["Missions_CreepScore"],
                    game1sum["participants"][3]["championName"],
                    game1sum["participants"][3]["challenges"]["turretPlatesTaken"],
                    game1sum["teams"][0]["objectives"]["baron"]["kills"],
                    game1sum["teams"][0]["objectives"]["dragon"]["kills"],
                    game1sum["teams"][0]["objectives"]["horde"]["kills"],
                    game1sum["teams"][0]["objectives"]["inhibitor"]["kills"],
                    game1sum["teams"][0]["objectives"]["riftHerald"]["kills"],
                    game1sum["teams"][0]["objectives"]["tower"]["kills"], game1sum["gameVersion"]))
        Supp.extend((game1sum["participants"][9]["championName"], game1sum["teams"][1]["objectives"]["baron"]["kills"],
                     game1sum["teams"][1]["objectives"]["dragon"]["kills"],
                     game1sum["teams"][1]["objectives"]["horde"]["kills"],
                     game1sum["teams"][1]["objectives"]["inhibitor"]["kills"],
                     game1sum["teams"][1]["objectives"]["riftHerald"]["kills"],
                     game1sum["teams"][1]["objectives"]["tower"]["kills"],
                     str(datetime.timedelta(seconds=game1sum["participants"][0]["challenges"]["gameLength"])),
                     game1sum["participants"][9]["challenges"]["turretPlatesTaken"],
                     game1sum["participants"][9]["challenges"]["teamDamagePercentage"],
                     game1sum["participants"][9]["magicDamageDealtToChampions"],
                     game1sum["participants"][9]["physicalDamageDealtToChampions"],
                     game1sum["participants"][9]["totalDamageDealtToChampions"],
                     game1sum["participants"][9]["goldEarned"], game1sum["participants"][9]["kills"],
                     game1sum["participants"][9]["deaths"],
                     game1sum["participants"][9]["assists"], game1sum["participants"][9]["visionScore"],
                     game1sum["participants"][9]["challenges"]["controlWardsPlaced"],
                     game1sum["participants"][9]["missions"]["Missions_CreepScore"],
                     game1sum["participants"][4]["championName"],
                     game1sum["participants"][4]["challenges"]["turretPlatesTaken"],
                     game1sum["teams"][0]["objectives"]["baron"]["kills"],
                     game1sum["teams"][0]["objectives"]["dragon"]["kills"],
                     game1sum["teams"][0]["objectives"]["horde"]["kills"],
                     game1sum["teams"][0]["objectives"]["inhibitor"]["kills"],
                     game1sum["teams"][0]["objectives"]["riftHerald"]["kills"],
                     game1sum["teams"][0]["objectives"]["tower"]["kills"], game1sum["gameVersion"]))

    # Appends the teamname of the scrim opponent
    if game1["teams"][0]["name"] == TeamID:
        Top.extend((game1["teams"][1]["name"]))
        Jungle.extend((game1["teams"][1]["name"]))
        Mid.extend((game1["teams"][1]["name"]))
        Bot.extend((game1["teams"][1]["name"]))
        Supp.extend((game1["teams"][1]["name"]))
    else:
        Top.extend((game1["teams"][0]["name"]))
        Jungle.extend((game1["teams"][0]["name"]))
        Mid.extend((game1["teams"][0]["name"]))
        Bot.extend((game1["teams"][0]["name"]))
        Supp.extend((game1["teams"][0]["name"]))

    # Creates the neccessary rows and writes the data into the sheet
    space = len(wks.get("A:A")) + 1
    wks.update('A'+ str(space)+ ':AJ'+ str(space+5),[Top, Jungle, Mid, Bot, Supp])



<ipython-input-1-91520dd9bcc1>:340: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  wks.update('A'+ str(space)+ ':AJ'+ str(space+5),[Top, Jungle, Mid, Bot, Supp])
